In [ ]:
import numpy as np

R = np.array([[0,2,3,0],[4,0,2,0],[0,2,0,2],[1,0,3,0]], dtype=float) #definimos nuestra matriz estocastica 
n = R.shape[0] #estados
r = np.max(R.sum(axis=1))#calculamos el maximo de la suma de la iesima fila
P = np.zeros((n,n))#matriz p sombrerito

for i in range(n):
    ri = R[i].sum() #tasa total de salida
    for j in range(n): #llenamos la matirz 
        if i==j:
            P[i,j] = 1 - ri/r #diagonal
        else:
            P[i,j] = R[i,j]/r

def Pt(t, M):
    L = r*t #parametro de poisson
    w = np.zeros(M+1) 
    w[0] = np.exp(-L) 
    for k in range(1,M+1): 
        w[k] = w[k-1]*(L/k) #pesos de poissson
    res = np.zeros((n,n)) #acumula la suma
    A = np.eye(n) #potencia
    for k in range(M+1):
        res += w[k]*A
        if k<M:
            A = A@P
    return res #devolvemos la matrixz aproximada

def Peps(t, e=1e-5): #acumulamos términos hasta que la masa de probabilidad de Poisson alcanza 1-e
    L = r*t
    A = np.eye(n)
    B = np.exp(-L)*np.eye(n)
    c = np.exp(-L)
    s = c
    k = 1
    while s < 1 - e:
        c = c*(L/k)
        A = A@P
        B = B + c*A
        s += c
        k += 1
    return B, k-1

def Mh(t): #calculamos el numero de terminos euristicos
    L = r*t
    return max(int(np.ceil(L + 5*np.sqrt(L))), 20)

def mostrar(M, dec=6):
    for fila in M:
        print("  " + "  ".join([f"{x:>{dec+2}.{dec}f}" for x in fila]))
        
def menu():
    print("Que ejercicio programable desea hacer ??")
    print("3.-Heuristico")
    print("4.-Adaptativo")
    print("0.-Salir")

while True:
    menu()
    op = input("Opcion: ")
    
    if op == '0':
        print("Fin.")
        break
    elif op == '3':
        print("\n--- Metodo heuristico ---")
        for t in [0.5, 1.0, 5.0]:
            M = Mh(t)
            print(f"\nt = {t}, M = {M}")
            mostrar(Pt(t, M), 6)
        # Chapman-Kolmogorov
        P05 = Pt(0.5, Mh(0.5))
        P1 = Pt(1.0, Mh(1.0))
        dif = np.max(np.abs(P1 - P05 @ P05))
        print(f"\nDif max |P(1) - P(0.5)^2| = {dif:.2e}")
    elif op == '4':
        print("\n--- Metodo adaptativo (epsilon=1e-5) ---")
        for t in [0.5, 1.0, 5.0]:
            B, Mf = Peps(t, 1e-5)
            print(f"\nt = {t}, M usado = {Mf}")
            mostrar(B, 6)
        print("\n--- Comparacion con heuristico ---")
        for t in [0.5, 1.0, 5.0]:
            B, _ = Peps(t, 1e-5)
            H = Pt(t, Mh(t))
            dif = np.max(np.abs(B - H))
            print(f"t = {t}: diferencia maxima = {dif:.2e}")
    else:
        print("Opcion invalida.")



Que ejercicio programable desea hacer ??
3.-Heuristico
4.-Adaptativo
0.-Salir
